# Task 4.3. Thử nghiệm theo bài báo trên GGNN và Graph B

| | |
|---|---|
| **Mục đích** | Đo tác động của hai can thiệp lấy từ bài báo, so với nhóm đối chứng chạy trong cùng phiên |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` (version 1) |
| **Đầu ra** | `results_4_3/<ARM>/`, `token_init_4_3/`, thư mục `submission_4_3` cùng tệp zip để nộp |
| **Máy chạy** | Kaggle GPU (T4 hoặc P100), bật Internet để tải CodeBERT |

**Thiết kế khác Task 4.1b và 4.2.** Hai task trước là bài toán *chọn cấu hình*: nhiều cấu hình, chọn bằng validation, chỉ chấm test cho cấu hình thắng. Task 4.3 là bài toán *đo tác động*: ba nhánh được đăng ký trước cùng giả thuyết, mỗi nhánh chạy ba seed, và test được báo cáo cho cả ba. Không nhánh nào được chọn làm mô hình của nhóm dựa trên điểm test.

| Nhánh | Thay đổi so với C0 | Giả thuyết |
|---|---|---|
| **C0** | Nhóm đối chứng, cấu hình L1 của Task 4.2 | Mốc so sánh, huấn luyện lại trong cùng phiên |
| **P1** | Khởi tạo embedding token bằng CodeBERT | CodeBERT cấp ngữ nghĩa mã nguồn mà vocabulary tự học không có |
| **P2** | Loss thích nghi kiểu LIVABLE | Nâng F1 của các lớp CWE ít mẫu, nhờ đó nâng Macro-F1 |

**Vì sao phải chạy lại nhóm đối chứng.** Task 4.2 cho thấy cùng cấu hình B0, cùng seed, cùng phiên bản torch nhưng chạy ở hai phiên khác nhau cho Macro-F1 test lệch 0,011, ngang với mọi chênh lệch ta đang cố đo. Vì vậy C0 phải chạy cùng phiên với P1 và P2; số của Task 4.2 chỉ dùng làm tham chiếu lịch sử.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
APPROVED_BRANCH = "feature/4_3_paper_experiments"
APPROVED_COMMIT = "21c3dac8ac9c1c906481ceb30230252bf5bba529"  # SHA đầy đủ của commit chứa mã nguồn
DATASET_ID = "vtnmai/diversevul-9077-processed-graph-data/versions/1"
APPROVED_ARMS = ["C0", "P1", "P2"]  # đăng ký trước, không thêm bớt sau khi thấy kết quả
CODEBERT_NAME = "microsoft/codebert-base"
RUN_EXPERIMENTS = True
ACTIVE = RUN_EXPERIMENTS  # các ô chuẩn bị: clone, kiểm tra dataset, xuất embedding
# Phục hồi sau khi phiên chết: gắn output cũ làm dataset rồi trỏ tới thư mục kết quả đã lưu.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_3
OUT_DIR = Path("/kaggle/working/results_4_3")
INIT_DIR = Path("/kaggle/working/token_init_4_3")
FIG_DIR = Path("/kaggle/working/figures_4_3")
SUBMISSION_DIR = Path("/kaggle/working/submission_4_3")
if RUN_EXPERIMENTS:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
    if not re.fullmatch(r"vtnmai/diversevul-9077-processed-graph-data/versions/[1-9][0-9]*", DATASET_ID):
        raise ValueError("Ghi phiên bản Kaggle dataset đã gắn vào notebook.")
REPO_DIR = Path("/tmp/g43") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR

## 1. Tải mã nguồn và ghi phiên bản

Clone nhánh đã duyệt, checkout đúng SHA. Giữ output notebook để lưu nhật ký chạy.

In [ ]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training import paper_experiments as arms
    from src.training.gcn_tuning import code_identity, environment
    from src.graph.processed import ids_hash, load_schema
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings (T4 hoặc P100)."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    assert set(APPROVED_ARMS) == set(arms.ARMS), "Danh sách nhánh khác định nghĩa trong mã nguồn."
    print("Revision:", actual, "Environment:", environment(), flush=True)


def run_script(script, arguments, log_path):
    cmd = [sys.executable, script, *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")

## 2. Dữ liệu và kế hoạch đã đăng ký

Kiểm tra dataset giống Task 4.2, rồi in ra cấu hình và giả thuyết của từng nhánh **trước khi chạy**. Mỗi nhánh chỉ đổi đúng một yếu tố so với C0.

In [ ]:
if ACTIVE:
    found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
             if all((p.parent / protocol / "vocab.json").is_file() for protocol in arms.PROTOCOLS)]
    assert len(found) == 1, "Gắn đúng một phiên bản dataset processed chính thức."
    PROCESSED = found[0]
    metadata = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    assert metadata.get("spec_version") in {"1.1", "1.2"} and metadata.get("n_valid_samples") == 7799
    assert sorted(load_schema(REPO_DIR / "configs/graph/schema_spec.yaml")["cwe_mapping"].values()) == list(range(23))
    from collections import Counter
    expected = {"project_wise": {"train": 6101, "validation": 762, "test": 936},
                "seen_project": {"train": 6272, "validation": 764, "test": 763}}
    VOCAB = {}
    for protocol in arms.PROTOCOLS:
        with (PROCESSED / protocol / "manifest.csv").open(encoding="utf-8", newline="") as handle:
            manifest = list(csv.DictReader(handle))
        assert len({row["sample_id"] for row in manifest}) == len(manifest)
        assert all(0 <= int(row["y"]) < 23 for row in manifest)
        vocab = json.loads((PROCESSED / protocol / "vocab.json").read_text(encoding="utf-8"))
        assert vocab["token"].get("<PAD>") == 0 and vocab["token"].get("<UNK>") == 1
        assert vocab.get("train_ids_sha256") == ids_hash(r["sample_id"] for r in manifest if r["split"] == "train")
        assert dict(Counter(row["split"] for row in manifest)) == expected[protocol]
        for split in expected[protocol]:
            assert (PROCESSED / protocol / f"graph_b_{split}.pt").is_file()
        VOCAB[protocol] = vocab
        print(f"{protocol}: {len(vocab['token'])} token, K={vocab['k']}")
    print("Dataset:", DATASET_ID, PROCESSED, flush=True)

    control = arms.arm_configuration("C0")
    print("\nKế hoạch đã đăng ký trước khi chạy:")
    for arm in APPROVED_ARMS:
        changed = {k: v for k, v in arms.arm_configuration(arm).items() if v != control[k]}
        assert len(changed) <= 1, f"{arm} đổi nhiều hơn một yếu tố"
        print(f"  {arm}: thay đổi {changed or 'không, đây là nhóm đối chứng'}")
        print(f"      giả thuyết: {arms.ARM_HYPOTHESES[arm]}")
    print(f"\nSố lần huấn luyện: {len(APPROVED_ARMS)} nhánh x 2 protocol x 3 seed = "
          f"{len(APPROVED_ARMS) * 6}")

## 3. Xuất bảng khởi tạo embedding token từ CodeBERT

Mỗi token trong vocabulary được đưa qua tokenizer của CodeBERT, lấy trung bình embedding đầu vào của các sub-token, rồi chiếu xuống 128 chiều bằng SVD rút gọn và chuẩn hóa về độ lệch chuẩn 1. Kiến trúc mô hình không đổi, chỉ giá trị khởi tạo của bảng embedding token khác đi, và bảng vẫn được huấn luyện tiếp.

Bước này cần `transformers` và Internet. Kết quả được lưu thành tệp tensor, nên mã huấn luyện không phụ thuộc `transformers`.

In [ ]:
if ACTIVE:
    INIT_DIR.mkdir(parents=True, exist_ok=True)
    saved_init = READ_ROOT.parent / INIT_DIR.name if SAVED_RESULTS_ROOT else INIT_DIR
    TOKEN_INIT = {p: INIT_DIR / f"{p}_dim128.pt" for p in arms.PROTOCOLS}
    if not all(path.is_file() for path in TOKEN_INIT.values()):
        for protocol, path in TOKEN_INIT.items():
            previous = saved_init / path.name
            if previous.is_file() and not path.is_file():
                shutil.copy2(previous, path)
    if not all(path.is_file() for path in TOKEN_INIT.values()):
        import transformers  # dừng sớm với lỗi rõ ràng nếu thiếu thư viện
        print("transformers", transformers.__version__, flush=True)
        run_script("scripts/export_token_embeddings.py",
                   ["--processed-dir", str(PROCESSED), "--out-dir", str(INIT_DIR),
                    "--protocols", *arms.PROTOCOLS, "--model-name", CODEBERT_NAME, "--dim", "128"],
                   INIT_DIR / "export.log")
    for protocol, path in TOKEN_INIT.items():
        payload = torch.load(path, map_location="cpu", weights_only=True)
        assert payload["protocol"] == protocol and payload["model_name"] == CODEBERT_NAME
        assert payload["matrix"].shape == (len(VOCAB[protocol]["token"]), 128)
        assert torch.isfinite(payload["matrix"]).all() and payload["matrix"][0].abs().sum() == 0
        print(f"{protocol}: {tuple(payload['matrix'].shape)}, std={payload['matrix'].std():.3f}")

### Kiểm tra khởi tạo có mang ngữ nghĩa thật không

Trước khi huấn luyện, xem các token gần nhau nhất trong không gian khởi tạo. Nếu CodeBERT thực sự mang thông tin, các token liên quan phải nằm gần nhau; nếu danh sách trông ngẫu nhiên thì giả thuyết P1 đã sai ngay từ đầu vào chứ không phải do mô hình.

In [ ]:
if ACTIVE:
    import torch.nn.functional as F
    protocol = "project_wise"
    payload = torch.load(TOKEN_INIT[protocol], map_location="cpu", weights_only=True)
    tokens = [t for t, _ in sorted(VOCAB[protocol]["token"].items(), key=lambda kv: kv[1])]
    normed = F.normalize(payload["matrix"], dim=1)
    probes = [t for t in ("malloc", "free", "memcpy", "strlen", "if", "int", "buf", "len")
              if t in VOCAB[protocol]["token"]]
    print(f"Token gần nhất theo cosine trong không gian khởi tạo CodeBERT ({protocol}):\n")
    for probe in probes:
        index = VOCAB[protocol]["token"][probe]
        scores = normed @ normed[index]
        scores[index] = -2.0
        top = scores.topk(5).indices.tolist()
        print(f"  {probe:>10} -> " + ", ".join(f"{tokens[j]} ({scores[j]:.2f})" for j in top))

## 4. Chạy ba nhánh

Mỗi nhánh gọi `scripts/train_gnn.py` với tham số do `paper_experiments.arm_arguments` sinh ra, nên cấu hình trong notebook không thể lệch khỏi định nghĩa trong mã nguồn. Nhánh đã chạy xong được xác minh rồi dùng lại, không chạy lại.

Khác hai task trước, mỗi nhánh đánh giá cả validation lẫn test trong một lần chạy, vì thiết kế là đo tác động chứ không phải chọn cấu hình.

In [ ]:
if RUN_EXPERIMENTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    plan = {"code": code_identity(REPO_DIR), "dataset_id": DATASET_ID,
            "model": arms.SCOPE[0], "graph": arms.SCOPE[1], "base_config_id": arms.BASE_CONFIG_ID,
            "arms": {a: arms.arm_configuration(a) for a in APPROVED_ARMS},
            "hypotheses": {a: arms.ARM_HYPOTHESES[a] for a in APPROVED_ARMS},
            "protocols": list(arms.PROTOCOLS), "seeds": list(arms.SEEDS),
            "design": "pre-registered arms; test reported for every arm; no selection by test"}
    plan_path = OUT_DIR / "experiment_plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:
            raise ValueError("Kế hoạch đã lưu khác cấu hình hiện tại; không ghi đè.")
    else:
        with plan_path.open("x", encoding="utf-8") as handle:
            json.dump(plan, handle, ensure_ascii=False, indent=2)
    for arm in APPROVED_ARMS:
        target = OUT_DIR / arm
        source = target if target.exists() else READ_ROOT / arm
        if source.exists():
            arms.validate_arm(source, arm)
            if source.resolve() != target.resolve():
                shutil.copytree(source, target)
            print(f"{arm}: dùng lại kết quả đã có", flush=True)
            continue
        # Mỗi protocol có vocabulary riêng nên P1 nhận một tệp khởi tạo cho mỗi protocol.
        extra = arms.arm_arguments(arm, [TOKEN_INIT[p] for p in arms.PROTOCOLS] if arm == "P1" else None)
        run_script("scripts/train_gnn.py",
                   ["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                    "--models", arms.SCOPE[0], "--graphs", arms.SCOPE[1],
                    "--protocols", *arms.PROTOCOLS, "--seeds", *map(str, arms.SEEDS),
                    "--config-id", arm, "--dataset-id", DATASET_ID, *extra],
                   OUT_DIR / f"{arm}.log")
        arms.validate_arm(target, arm)

## 5. Kết quả: tác động của từng can thiệp

Bảng và biểu đồ dưới đây so mỗi nhánh với **C0 chạy trong cùng phiên**. Kết luận "có tác động" chỉ được đưa ra khi trị tuyệt đối của chênh lệch vượt độ lệch chuẩn lớn hơn giữa hai nhóm.

In [ ]:
if RUN_EXPERIMENTS:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    COLORS = {"C0": "#7570b3", "P1": "#1b9e77", "P2": "#d95f02"}
    LABEL = arms.ARM_LABELS
    res = pd.concat([pd.read_csv(OUT_DIR / a / "results.csv").assign(arm=a) for a in APPROVED_ARMS],
                    ignore_index=True)
    per_class = pd.concat([pd.read_csv(OUT_DIR / a / "per_class.csv").assign(arm=a) for a in APPROVED_ARMS],
                          ignore_index=True)
    hist = {}
    for a in APPROVED_ARMS:
        for item in json.loads((OUT_DIR / a / "training_history.json").read_text(encoding="utf-8")):
            protocol, _, _, seed = item["tag"].split("|")
            hist[(a, protocol, int(seed))] = pd.DataFrame(item["history"])

    table = pd.DataFrame(index=[LABEL[a] for a in APPROVED_ARMS])
    for split in ("validation", "test"):
        for protocol in arms.PROTOCOLS:
            sub = res[(res.split == split) & (res.protocol == protocol)].groupby("arm").macro_f1
            table[f"{split} {protocol}"] = [
                f"{sub.mean()[a]:.4f} ± {sub.std(ddof=0)[a]:.4f}" for a in APPROVED_ARMS]
    test_pw = res[(res.split == "test") & (res.protocol == "project_wise")].groupby("arm")
    for metric in ("weighted_f1", "mcc", "accuracy"):
        table[f"test pw {metric}"] = [round(test_pw[metric].mean()[a], 4) for a in APPROVED_ARMS]
    table["giây/lần"] = [round(res[res.arm == a].train_seconds.mean()) for a in APPROVED_ARMS]
    display(table)
    table.to_csv(FIG_DIR / "arm_summary.csv")

In [ ]:
if RUN_EXPERIMENTS:
    rows = []
    for protocol in arms.PROTOCOLS:
        group = res[(res.split == "test") & (res.protocol == protocol)].groupby("arm").macro_f1
        mean, std = group.mean(), group.std(ddof=0)
        for arm in APPROVED_ARMS:
            if arm == "C0":
                continue
            delta = mean[arm] - mean["C0"]
            bound = max(std[arm], std["C0"])
            rows.append({"protocol": protocol, "nhánh": LABEL[arm],
                         "Macro-F1 test": round(mean[arm], 4), "C0": round(mean["C0"], 4),
                         "Δ": round(delta, 4), "độ lệch chuẩn lớn hơn": round(bound, 4),
                         "kết luận": "trong độ lệch chuẩn" if abs(delta) < bound
                         else ("cao hơn C0" if delta > 0 else "thấp hơn C0")})
    effects = pd.DataFrame(rows)
    display(effects)
    effects.to_csv(FIG_DIR / "effect_vs_control.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, protocol in zip(axes, arms.PROTOCOLS):
        sub = res[(res.split == "test") & (res.protocol == protocol)]
        mean = sub.groupby("arm").macro_f1.mean().reindex(APPROVED_ARMS)
        std = sub.groupby("arm").macro_f1.std(ddof=0).reindex(APPROVED_ARMS)
        ax.bar([LABEL[a] for a in APPROVED_ARMS], mean, yerr=std, capsize=4,
               color=[COLORS[a] for a in APPROVED_ARMS])
        for i, arm in enumerate(APPROVED_ARMS):
            ax.scatter([i] * 3, sub[sub.arm == arm].macro_f1, color="black", s=12, zorder=3)
        ax.axhline(mean["C0"], color="gray", ls="--", lw=1, label="C0 đối chứng")
        ax.set_title(f"Macro-F1 test, {protocol}")
        ax.tick_params(axis="x", rotation=12)
        ax.legend()
    axes[0].set_ylabel("Macro-F1 test (trung bình ± độ lệch chuẩn, chấm = từng seed)")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "effect_test_macro_f1.png", bbox_inches="tight")
    plt.show()

### Kiểm chứng trực tiếp giả thuyết của P2

P2 đặt cược vào các lớp CWE ít mẫu. Bảng dưới chia 23 lớp theo số mẫu test thành nhóm đầu, giữa và đuôi, giống cách phân nhóm của LIVABLE. Nếu P2 đúng, F1 nhóm đuôi phải tăng; nếu chỉ Macro-F1 tổng tăng mà nhóm đuôi không tăng thì giả thuyết chưa được chứng minh.

In [ ]:
if RUN_EXPERIMENTS:
    protocol = "project_wise"
    pc = per_class[(per_class.split == "test") & (per_class.protocol == protocol)]
    support = pc[pc.arm == "C0"].groupby("cwe").support.first()
    def bucket(n):
        return "đầu (≥50)" if n >= 50 else ("giữa (16-49)" if n >= 16 else "đuôi (≤15)")
    groups = support.map(bucket)
    mean_f1 = pc.groupby(["arm", "cwe"]).f1.mean().unstack(0)[APPROVED_ARMS]
    summary = mean_f1.groupby(groups).agg(["mean", "size"]).round(4)
    summary.columns = [f"{a} {s}" for a, s in summary.columns]
    summary = summary[[c for c in summary.columns if not c.endswith("size")] + [f"C0 size"]]
    summary = summary.rename(columns={"C0 size": "số CWE"})
    display(summary)
    zero = (mean_f1 == 0).sum()
    print("Số CWE có F1 test bằng 0 trên 23:", dict(zip(zero.index, zero.values)))
    summary.to_csv(FIG_DIR / "per_cwe_groups.csv")

    ordered = mean_f1.loc[support.sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(13, 4.4))
    x = np.arange(len(ordered))
    width = 0.27
    for i, arm in enumerate(APPROVED_ARMS):
        ax.bar(x + (i - 1) * width, ordered[arm], width, label=LABEL[arm], color=COLORS[arm])
    ax.set_xticks(x)
    ax.set_xticklabels(ordered.index, rotation=70, ha="right")
    ax.set_ylabel("F1 theo CWE (trung bình 3 seed)")
    twin = ax.twinx()
    twin.plot(x, support.loc[ordered.index], color="black", marker="o", ms=3, lw=1)
    twin.set_ylabel("Số mẫu test")
    twin.grid(False)
    ax.set_title(f"F1 theo CWE trên test {protocol}, sắp theo support giảm dần")
    ax.legend(loc="upper right")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "per_cwe_by_arm.png", bbox_inches="tight")
    plt.show()

### Đường học và dấu hiệu học thuộc

P2 đổi hàm mất mát nên loss train của nó không so trực tiếp được với C0 và P1. Vì vậy biểu đồ loss chỉ vẽ C0 và P1; Macro-F1 validation vẽ cả ba vì cùng thang đo.

In [ ]:
if RUN_EXPERIMENTS:
    fig, axes = plt.subplots(2, len(arms.PROTOCOLS), figsize=(12, 7), sharex="col")
    for j, protocol in enumerate(arms.PROTOCOLS):
        for arm in APPROVED_ARMS:
            for seed in arms.SEEDS:
                frame = hist[(arm, protocol, seed)]
                label = LABEL[arm] if seed == arms.SEEDS[0] else None
                if arm != "P2":  # loss khác thang đo
                    axes[0, j].plot(frame.epoch, frame.train_loss, color=COLORS[arm], alpha=0.7, label=label)
                axes[1, j].plot(frame.epoch, frame.val_macro_f1, color=COLORS[arm], alpha=0.7, label=label)
        axes[0, j].set_title(f"Loss train, {protocol} (P2 khác hàm mất mát nên không vẽ)")
        axes[1, j].set_title(f"Macro-F1 validation, {protocol}")
        axes[1, j].set_xlabel("Epoch")
        axes[0, j].legend(fontsize=8)
        axes[1, j].legend(fontsize=8)
    axes[0, 0].set_ylabel("train loss")
    axes[1, 0].set_ylabel("Macro-F1 validation")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "learning_curves.png", bbox_inches="tight")
    plt.show()
    overview = []
    for arm in APPROVED_ARMS:
        for protocol in arms.PROTOCOLS:
            sub = res[(res.arm == arm) & (res.protocol == protocol) & (res.split == "test")]
            peak = sub.peak_gpu_mb.max()  # trống khi chạy trên CPU
            overview.append({"nhánh": LABEL[arm], "protocol": protocol,
                             "epoch tốt nhất": round(sub.best_epoch.mean(), 1),
                             "epoch đã chạy": round(sub.epochs_run.mean(), 1),
                             "loss train cuối": round(np.mean(
                                 [hist[(arm, protocol, s)].train_loss.iloc[-1] for s in arms.SEEDS]), 4),
                             "VRAM đỉnh (MB)": round(peak) if pd.notna(peak) else None})
    display(pd.DataFrame(overview))

## 6. Đặt cạnh kết quả các task trước

Các số của Task 4.1a, 4.1b và 4.2 đến từ những phiên chạy khác, nên chỉ dùng làm **tham chiếu lịch sử**. Kết luận về tác động của P1 và P2 chỉ dựa trên so sánh với C0 trong cùng phiên ở mục 5.

In [ ]:
if RUN_EXPERIMENTS:
    A = REPO_DIR / "reports/graph/4_1a_results"
    history_rows = []
    base = pd.read_csv(A / "baselines_results.csv")
    gnn = pd.read_csv(A / "gnn_results.csv")
    for model in ("token_nodes", "token_method", "random"):
        history_rows.append(base[(base.split == "test") & (base.model == model)].assign(method=f"{model} (4.1a)"))
    for model in ("gcn", "ggnn"):
        for graph in ("graph_a", "graph_b"):
            history_rows.append(gnn[(gnn.split == "test") & (gnn.model == model) & (gnn.graph == graph)]
                                .assign(method=f"{model.upper()}-{graph[-1].upper()} mặc định (4.1a)"))
    history_rows.append(pd.read_csv(REPO_DIR / "reports/graph/4_1b_results/final/results.csv")
                        .assign(method="GCN-A tinh chỉnh L1 (4.1b)"))
    history_rows.append(pd.read_csv(REPO_DIR / "reports/graph/4_2_results/final/results.csv")
                        .assign(method="GGNN-B tinh chỉnh L1 (4.2)"))
    history_rows.append(pd.read_csv(REPO_DIR / "reports/graph/4_2_results/sensitivity/B0/results.csv")
                        .assign(method="GGNN-B B0 (4.2, độ nhạy)"))
    for arm in APPROVED_ARMS:
        history_rows.append(res[(res.arm == arm) & (res.split == "test")].assign(method=f"{LABEL[arm]} (4.3)"))
    allr = pd.concat(history_rows, ignore_index=True)
    methods = list(dict.fromkeys(allr.method))
    overall = pd.DataFrame(index=methods)
    for protocol in arms.PROTOCOLS:
        sub = allr[allr.protocol == protocol].groupby("method").macro_f1
        overall[f"{protocol} Macro-F1"] = sub.apply(
            lambda s: f"{s.mean():.4f} ± {s.std(ddof=0):.4f}").reindex(methods)
    display(overall)
    overall.to_csv(FIG_DIR / "history_comparison.csv")

    fig, ax = plt.subplots(figsize=(12, 4.4))
    width = 0.38
    for i, protocol in enumerate(arms.PROTOCOLS):
        sub = allr[allr.protocol == protocol].groupby("method").macro_f1
        ax.bar(np.arange(len(methods)) + (i - 0.5) * width, sub.mean().reindex(methods), width,
               yerr=sub.std(ddof=0).reindex(methods), capsize=3, label=protocol)
    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels(methods, rotation=30, ha="right")
    ax.set_ylabel("Macro-F1 test (trung bình ± độ lệch chuẩn)")
    ax.set_title("Task 4.3 đặt cạnh các mốc trước đó")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIG_DIR / "history_comparison.png", bbox_inches="tight")
    plt.show()

In [ ]:
if RUN_EXPERIMENTS:
    matrices = {a: json.loads((OUT_DIR / a / "confusion_matrices.json").read_text(encoding="utf-8"))
                for a in APPROVED_ARMS}
    labels = list(per_class[(per_class.arm == "C0") & (per_class.protocol == "project_wise")
                            & (per_class.seed == arms.SEEDS[0]) & (per_class.split == "test")].cwe)
    fig, axes = plt.subplots(1, len(APPROVED_ARMS), figsize=(19, 6.2))
    for ax, arm in zip(axes, APPROVED_ARMS):
        key = f"project_wise|{arms.SCOPE[1]}|{arms.SCOPE[0]}|{arms.SEEDS[0]}|test"
        cm = np.array(matrices[arm][key], dtype=float)
        norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=6)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=6)
        ax.set_xlabel("Dự đoán")
        ax.set_title(f"{LABEL[arm]}, test project_wise, seed {arms.SEEDS[0]}")
        ax.grid(False)
    axes[0].set_ylabel("Thực tế")
    fig.colorbar(im, ax=axes, fraction=0.012)
    fig.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
    plt.show()

## 7. Đóng gói để nộp

Gom tệp nhẹ vào `submission_4_3` và tệp zip cùng tên. Sau khi chạy xong: bấm **Save Version** (Save & Run All) để lưu bản notebook chứa log, rồi tải tệp zip và notebook đã chạy từ tab Output.

In [ ]:
if RUN_EXPERIMENTS:
    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    SUBMISSION_DIR.mkdir(parents=True)
    light = shutil.ignore_patterns("*.pt")
    for item in sorted(OUT_DIR.iterdir()):
        if item.is_dir():
            shutil.copytree(item, SUBMISSION_DIR / "arms" / item.name, ignore=light)
        elif item.suffix in {".json", ".log"}:
            (SUBMISSION_DIR / "arms").mkdir(exist_ok=True)
            shutil.copy2(item, SUBMISSION_DIR / "arms" / item.name)
    shutil.copytree(FIG_DIR, SUBMISSION_DIR / "figures")
    (SUBMISSION_DIR / "token_init").mkdir()
    for path in sorted(INIT_DIR.glob("*.log")):
        shutil.copy2(path, SUBMISSION_DIR / "token_init" / path.name)
    for protocol, path in TOKEN_INIT.items():  # chỉ giữ dấu vân tay, không giữ tensor nặng
        payload = torch.load(path, map_location="cpu", weights_only=True)
        import hashlib
        (SUBMISSION_DIR / "token_init" / f"{protocol}.json").write_text(json.dumps(
            {"model_name": payload["model_name"], "dim": payload["dim"],
             "n_tokens": payload["n_tokens"],
             "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}, indent=2), encoding="utf-8")
    archive = shutil.make_archive(str(SUBMISSION_DIR), "zip", SUBMISSION_DIR)
    print(f"Đã tạo {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
    for p in sorted(SUBMISSION_DIR.rglob("*")):
        if p.is_file():
            print(" ", p.relative_to(SUBMISSION_DIR))